<a href="https://colab.research.google.com/github/mervefilizbaker1/NLP-Homework-02-Word-Embeddings/blob/main/NLP_HW02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [ ]:
!pip install gensim

In [ ]:
!pip install nltk

# Libraries

In [ ]:
from datasets import load_dataset
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')
import os

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
dataset = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train")

# Dataset

In [ ]:
print(dataset[0]["text"][:200])

April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.

April always begins on the same day 


In [ ]:
print(len(dataset))

241787


# 2.2 Training Word Embeddings

In [ ]:
first_thousand= dataset["text"][:1000]

In [ ]:
all_sentences = []

In [ ]:
for article in first_thousand:
  sentences = sent_tokenize(article)
  for sentence in sentences:
    tokens = [token.lower() for token in word_tokenize(sentence) if token.isalpha()]
    all_sentences.append(tokens)

In [ ]:
print(len(all_sentences))
print(all_sentences[0])

39976
['april', 'apr']


In [ ]:
if os.path.exists("cbow_model.model"):
    os.remove("cbow_model.model")
    print("Removed existing cbow_model.model to prevent incompatibility issues.")

model_cbow = Word2Vec(
    sentences=all_sentences,
    sg= 0,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)
model_cbow.save("cbow_model.model")

Removed existing cbow_model.model to prevent incompatibility issues.


In [ ]:
if os.path.exists("sgram_model.model"):
    os.remove("sgram_model.model")
    print("Removed existing sgram_model.model to prevent incompatibility issues.")

model_sgram = Word2Vec(
    sentences=all_sentences,
    sg= 1,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)
model_sgram.save("sgram_model.model")

Removed existing sgram_model.model to prevent incompatibility issues.


In [ ]:
print(model_cbow.wv.most_similar("king"))

[('queen', 0.9694399237632751), ('leader', 0.8871856331825256), ('president', 0.8781343102455139), ('emperor', 0.8723028302192688), ('pope', 0.8710253238677979), ('elizabeth', 0.8687583804130554), ('independent', 0.8678393959999084), ('elected', 0.8672634959220886), ('prince', 0.866521418094635), ('prime', 0.864966094493866)]


In [ ]:
print(model_sgram.wv.most_similar("king"))

[('queen', 0.8885180354118347), ('monarch', 0.8537761569023132), ('vi', 0.8503477573394775), ('elizabeth', 0.8390105366706848), ('son', 0.8337311148643494), ('mary', 0.831142783164978), ('prince', 0.8254013061523438), ('viii', 0.8163965940475464), ('edward', 0.8102068901062012), ('emperor', 0.8101574778556824)]


In [ ]:
all_articles = dataset["text"]

In [ ]:
for article in all_articles:
  sentences = sent_tokenize(article)
  for sentence in sentences:
    tokens = [token.lower() for token in word_tokenize(sentence) if token.isalpha()]
    all_sentences.append(tokens)

In [ ]:
print(len(all_sentences))
print(all_sentences[0])

2425742
['april', 'apr']


# 2.3 Comparing Word Embeddings

In [ ]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")
google = api.load("word2vec-google-news-300")

In [ ]:
glove_100 = api.load("glove-wiki-gigaword-100")
glove_300 = api.load("glove-wiki-gigaword-300")

In [ ]:
queries = [
    {"type": "similarity", "query": "guitar"},
    {"type": "similarity", "query": "actor"},
    {"type": "analogy", "positive": ["ankara", "germany"], "negative": ["turkey"]},
    {"type": "analogy", "positive": ["mother", "man"], "negative": ["woman"]},
    {"type": "analogy", "positive": ["algorithm", "biology"], "negative": ["computer"]},
]

In [ ]:
models = {
    "CBOW": model_cbow.wv,
    "Skip-gram": model_sgram.wv,
    "GloVe-100": glove_100,
    "GloVe-300": glove_300
}

for name, model in models.items():
    for q in queries:
        if q["type"] == "similarity":
            result = model.most_similar(q["query"])
        else:
            result = model.most_similar(positive=q["positive"], negative=q["negative"])
        print(f"\n{name} | {q['type']}: {q.get('query', q.get('positive'))}")
        print(result)


CBOW | similarity: guitar
[('bus', 0.984205424785614), ('flash', 0.9834664463996887), ('g', 0.982266902923584), ('vehicle', 0.9817975759506226), ('travelling', 0.9803901314735413), ('strike', 0.9795804023742676), ('investment', 0.9791892170906067), ('receiving', 0.9783531427383423), ('bottle', 0.9776164293289185), ('direct', 0.9762037992477417)]

CBOW | similarity: actor
[('actress', 0.9940345883369446), ('singer', 0.9832363128662109), ('musician', 0.9800742864608765), ('footballer', 0.9782897233963013), ('presenter', 0.9747005701065063), ('politician', 0.9735411405563354), ('wrestler', 0.9727575182914734), ('writer', 0.9679427742958069), ('births', 0.967929482460022), ('b', 0.9669913053512573)]

CBOW | analogy: ['ankara', 'germany']
[('portugal', 0.9480643272399902), ('saint', 0.9381096363067627), ('yugoslavia', 0.9380465745925903), ('petersburg', 0.9372280836105347), ('czechoslovakia', 0.9362252950668335), ('cabinet', 0.9326961636543274), ('constitutional', 0.931922197341919), ('alb

# 2.4 Bias in Word Embeddings

In [ ]:
!pip install wefe

In [ ]:
from wefe.query import Query
from wefe.word_embedding_model import WordEmbeddingModel
from wefe.metrics import WEAT

In [ ]:
christianity_words = ["christian", "church", "bible", "jesus", "christmas", "cross", "prayer", "cathedral"]
islam_words = ["muslim", "mosque", "quran", "allah", "ramadan", "imam", "islam", "mecca"]

In [ ]:
positive_words = ["love", "peace", "joy", "happy", "wonderful", "beautiful", "good", "pleasant"]
negative_words = ["war", "terror", "evil", "bad", "horrible", "violent", "terrible", "hate"]

In [ ]:
query = Query(
    target_sets=[christianity_words, islam_words],
    attribute_sets=[positive_words, negative_words],
    target_sets_names=["Christianity", "Islam"],
    attribute_sets_names=["Positive", "Negative"]
)

In [ ]:
models = [
    WordEmbeddingModel(model_cbow.wv, "CBOW"),
    WordEmbeddingModel(model_sgram.wv, "Skip-gram"),
    WordEmbeddingModel(glove_100, "GloVe-100"),
    WordEmbeddingModel(glove_300, "GloVe-300"),
]

In [ ]:
for model in models:
    result = WEAT().run_query(query, model, lost_vocabulary_threshold=0.5)
    print(f"\n{model.name}")
    print(result)


CBOW
{'query_name': 'Christianity and Islam wrt Positive and Negative', 'result': -0.10078148330960951, 'weat': -0.10078148330960951, 'effect_size': -0.574066929740987, 'p_value': nan}

Skip-gram
{'query_name': 'Christianity and Islam wrt Positive and Negative', 'result': 0.28013233883040295, 'weat': 0.28013233883040295, 'effect_size': 1.080900591062332, 'p_value': nan}

GloVe-100
{'query_name': 'Christianity and Islam wrt Positive and Negative', 'result': 1.1540158810094, 'weat': 1.1540158810094, 'effect_size': 1.7685541244018692, 'p_value': nan}

GloVe-300
{'query_name': 'Christianity and Islam wrt Positive and Negative', 'result': 1.05700358573813, 'weat': 1.05700358573813, 'effect_size': 1.673571685328266, 'p_value': nan}


# 2.5 Text Classification: Sparse vs. Dense

In [ ]:
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', 'label': 2}


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]
test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

In [ ]:
vectorizer = TfidfVectorizer(max_features=50000)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train,train_labels)

LogisticRegression(max_iter=1000)

In [ ]:
preds = clf.predict(X_test)
print(classification_report(test_labels, preds))

              precision    recall  f1-score   support

           0       0.67      0.35      0.46      3972
           1       0.59      0.77      0.67      5937
           2       0.54      0.58      0.56      2375

    accuracy                           0.59     12284
   macro avg       0.60      0.56      0.56     12284
weighted avg       0.61      0.59      0.58     12284



In [ ]:
import numpy as np

In [ ]:
def get_avg_embedding(text, model, vector_size):
    tokens = text.lower().split()
    vectors = []
    for token in tokens:
        if token in model:
            vectors.append(model[token])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)

In [ ]:
embedding_models = {
    "CBOW": (model_cbow.wv, 100),
    "Skip-gram": (model_sgram.wv, 100),
    "GloVe-100": (glove_100, 100),
    "GloVe-300": (glove_300, 300),
}

In [ ]:
for name, (emb_model, vec_size) in embedding_models.items():

    X_train_emb = np.array([get_avg_embedding(t, emb_model, vec_size) for t in train_texts])
    X_test_emb = np.array([get_avg_embedding(t, emb_model, vec_size) for t in test_texts])

    clf_emb = LogisticRegression(max_iter=1000)
    clf_emb.fit(X_train_emb, train_labels)

    preds_emb = clf_emb.predict(X_test_emb)
    print(f"\n Model B — {name}")
    print(classification_report(test_labels, preds_emb))


 Model B — CBOW
              precision    recall  f1-score   support

           0       0.48      0.08      0.14      3972
           1       0.52      0.60      0.55      5937
           2       0.27      0.54      0.36      2375

    accuracy                           0.42     12284
   macro avg       0.42      0.41      0.35     12284
weighted avg       0.46      0.42      0.38     12284


 Model B — Skip-gram
              precision    recall  f1-score   support

           0       0.50      0.14      0.21      3972
           1       0.53      0.65      0.58      5937
           2       0.32      0.54      0.40      2375

    accuracy                           0.46     12284
   macro avg       0.45      0.44      0.40     12284
weighted avg       0.48      0.46      0.43     12284


 Model B — GloVe-100
              precision    recall  f1-score   support

           0       0.59      0.52      0.55      3972
           1       0.61      0.59      0.60      5937
           2  